# 12.8 — Graph autoencoders

Graph autoencoders learn embeddings by reconstructing edges.

A graph autoencoder encodes nodes as vectors and decodes pairwise dot products into edge probabilities. Balanced positive and negative edge batches keep the sparse objective inspectable.

Save a copy to Drive to edit.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 12610
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(x):
    values = np.asarray(x, dtype=float)
    shifted = values - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def leaky_relu(x, slope=0.2):
    values = np.asarray(x, dtype=float)
    return np.where(values >= 0.0, values, slope * values)


def graph_to_arrays(graph):
    nodes = list(graph.nodes())
    adjacency = nx.to_numpy_array(graph, nodelist=nodes, dtype=float)
    labels = np.array([graph.nodes[node].get("label", 0) for node in nodes], dtype=int)
    features = np.array([graph.nodes[node].get("x", [0.0, 0.0, 0.0]) for node in nodes], dtype=float)
    return nodes, adjacency, features, labels


def add_features(graph, labels, noise=0.05, seed=0):
    rng = np.random.default_rng(seed)
    degrees = np.array([degree for _, degree in graph.degree()], dtype=float)
    degree_scale = max(float(degrees.max()), 1.0)
    for index, node in enumerate(graph.nodes()):
        label = int(labels[index])
        base = np.array([1.0 - label, label, degrees[index] / degree_scale], dtype=float)
        graph.nodes[node]["label"] = label
        graph.nodes[node]["x"] = base + rng.normal(0.0, noise, size=3)
    return graph


def make_toy_graph():
    graph = nx.Graph()
    graph.add_edges_from([(0, 1), (1, 2), (2, 3), (0, 2)])
    labels = np.array([0, 0, 1, 1], dtype=int)
    return add_features(graph, labels, noise=0.0, seed=1)


def make_karate_graph():
    graph = nx.karate_club_graph()
    labels = np.array([0 if graph.nodes[node]["club"] == "Mr. Hi" else 1 for node in graph.nodes()], dtype=int)
    return add_features(graph, labels, noise=0.03, seed=2)


def make_sbm_graph(sizes, p_in, p_out, noise, seed):
    probs = [[p_in, p_out], [p_out, p_in]]
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    labels = []
    for block, size in enumerate(sizes):
        labels.extend([block] * size)
    return add_features(graph, np.array(labels, dtype=int), noise=noise, seed=seed + 100)


def build_graph_ladder():
    return [
        {"name": "D1 toy 4-node", "graph": make_toy_graph(), "complexity": 1},
        {"name": "D2 karate club", "graph": make_karate_graph(), "complexity": 2},
        {"name": "D3 clean SBM", "graph": make_sbm_graph([18, 18], 0.34, 0.04, 0.07, 3), "complexity": 3},
        {"name": "D4 larger SBM", "graph": make_sbm_graph([35, 35], 0.20, 0.025, 0.10, 4), "complexity": 4},
        {"name": "D5 noisy overlap", "graph": make_sbm_graph([45, 45], 0.16, 0.075, 0.22, 5), "complexity": 5},
    ]


def class_train_mask(labels):
    mask = np.zeros(len(labels), dtype=bool)
    for label in np.unique(labels):
        candidates = np.where(labels == label)[0]
        count = max(1, len(candidates) // 4)
        mask[candidates[:count]] = True
    return mask


def centroid_predict(embeddings, labels, train_mask):
    classes = np.unique(labels)
    centroids = []
    for label in classes:
        rows = embeddings[(labels == label) & train_mask]
        centroids.append(rows.mean(axis=0))
    centroids = np.vstack(centroids)
    distances = ((embeddings[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return classes[np.argmin(distances, axis=1)]


def aligned_accuracy(predictions, labels):
    raw = float(np.mean(predictions == labels))
    flipped = float(np.mean(1 - predictions == labels))
    return max(raw, flipped)


def pairwise_auc(positive_scores, negative_scores):
    total = 0.0
    count = 0.0
    for positive in positive_scores:
        for negative in negative_scores:
            if positive > negative:
                total += 1.0
            elif positive == negative:
                total += 0.5
            count += 1.0
    return total / count


def sample_link_split(graph, seed=0, holdout_fraction=0.2):
    rng = np.random.default_rng(seed)
    edges = [tuple(sorted(edge)) for edge in graph.edges()]
    rng.shuffle(edges)
    holdout_count = max(1, int(len(edges) * holdout_fraction))
    positive_edges = edges[:holdout_count]
    train_graph = graph.copy()
    train_graph.remove_edges_from(positive_edges)
    non_edges = [tuple(sorted(edge)) for edge in nx.non_edges(graph)]
    rng.shuffle(non_edges)
    negative_edges = non_edges[:holdout_count]
    return train_graph, positive_edges, negative_edges


def spectral_embeddings(adjacency, dimensions=8):
    adjacency = np.asarray(adjacency, dtype=float)
    augmented = adjacency + np.eye(adjacency.shape[0])
    degrees = augmented.sum(axis=1)
    inv_sqrt = 1.0 / np.sqrt(np.maximum(degrees, 1e-12))
    normalized = inv_sqrt[:, None] * augmented * inv_sqrt[None, :]
    values, vectors = np.linalg.eigh(normalized)
    order = np.argsort(values)[::-1]
    keep = order[: min(dimensions, adjacency.shape[0])]
    return vectors[:, keep] * values[keep]


def plot_graph_panels(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(15, 3))
    for axis, result in zip(axes, results):
        graph = result["graph"]
        colors = result.get("colors")
        pos = nx.spring_layout(graph, seed=SEED)
        nx.draw_networkx_nodes(graph, pos, node_color=colors, cmap="coolwarm", node_size=55, ax=axis)
        nx.draw_networkx_edges(graph, pos, alpha=result.get("edge_alpha", 0.25), width=0.8, ax=axis)
        axis.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(5, 3))
    plt.plot([result["complexity"] for result in results], [result["metric"] for result in results], marker="o")
    plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
    plt.ylabel(metric_name)
    plt.xlabel("graph ladder rung")
    plt.grid(True, alpha=0.3)
    plt.show()

def gae_reconstruct(z, positive_edges, negative_edges, threshold=0.6):
    z = np.asarray(z, dtype=float)
    probabilities = sigmoid(z @ z.T)
    positive_scores = np.array([float(z[i] @ z[j]) for i, j in positive_edges])
    negative_scores = np.array([float(z[i] @ z[j]) for i, j in negative_edges])
    positive_loss = -np.log(sigmoid(positive_scores) + 1e-12)
    negative_loss = -np.log(1.0 - sigmoid(negative_scores) + 1e-12)
    loss = float(np.mean(np.concatenate([positive_loss, negative_loss])))
    reconstructed = probabilities >= threshold
    return probabilities, positive_scores, negative_scores, loss, reconstructed


def gae_link_auc(graph):
    train_graph, positive_edges, negative_edges = sample_link_split(graph, seed=SEED)
    nodes = list(train_graph.nodes())
    adjacency = nx.to_numpy_array(train_graph, nodelist=nodes, dtype=float)
    embeddings = spectral_embeddings(adjacency, dimensions=8)
    index = {node: i for i, node in enumerate(nodes)}
    probabilities = sigmoid(embeddings @ embeddings.T)
    positive_scores = [probabilities[index[i], index[j]] for i, j in positive_edges]
    negative_scores = [probabilities[index[i], index[j]] for i, j in negative_edges]
    auc = pairwise_auc(positive_scores, negative_scores)
    return auc, positive_edges, negative_edges, probabilities

## The concept, built once: dot-product reconstruction

The standard decoder is $\hat A_{ij}=\sigma(z_i^\top z_j)$ with binary cross-entropy on positive and sampled negative edges. The lesson positive score is $z_0^\top z_1=0.800$.

In [ ]:
z = np.array([[1.0, 0.0], [0.8, 0.0], [-1.0, 0.0], [0.0, 1.0]])
positive_edges = [(0, 1), (1, 3)]
negative_edges = [(0, 2), (2, 3)]
probabilities, positive_scores, negative_scores, loss, reconstructed = gae_reconstruct(z, positive_edges, negative_edges)
print("positive score", positive_scores[0])
print("probability", probabilities[0, 1])
print("balanced BCE", loss)
assert round(float(positive_scores[0]), 3) == 0.800
assert round(float(probabilities[0, 1]), 3) == 0.690
assert round(float(loss), 3) == 0.342

Thresholding at $0.6$ reconstructs $(0,1)$ but not $(0,2)$. The mini-batch is balanced: $2$ positive edges and $2$ sampled negatives.

In [ ]:
print("reconstruct (0,1)", bool(reconstructed[0, 1]))
print("reconstruct (0,2)", bool(reconstructed[0, 2]))
print("batch sizes", len(positive_edges), len(negative_edges))
assert bool(reconstructed[0, 1]) is True
assert bool(reconstructed[0, 2]) is False
assert len(positive_edges) == 2
assert len(negative_edges) == 2

## The dataset ladder

Family F11 uses one inline graph ladder for every topic: D1 is a 4-node toy graph, D2 is karate club, D3 is a stochastic block model, D4 is larger, and D5 is noisy/overlapping. The metric for this notebook is edge AUC.

In [ ]:
ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    nodes, adjacency, features, labels = graph_to_arrays(graph)
    counts = np.bincount(labels)
    print(rung["name"], "nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "features", features.shape, "classes", counts.tolist())
    print("sample labels", labels[:8].tolist())

## Run the same graph autoencoder scoring across D1-D5

In [ ]:
results = []
for rung in ladder:
    graph = rung["graph"]
    auc, positive_edges, negative_edges, probabilities = gae_link_auc(graph)
    colors = graph_to_arrays(graph)[3]
    results.append({"name": rung["name"], "complexity": rung["complexity"], "graph": graph, "metric": auc, "colors": colors, "positive_edges": positive_edges})
    print(f"{rung['name']:<18} edge-AUC={auc:.3f} heldout={len(positive_edges)}")

## Results visualization

In [ ]:
plot_graph_panels(results, "AUC")

## Pitfall on D5: every non-edge swamps the loss

Sparse graphs have many more absent pairs than observed edges. Treating every non-edge as a negative changes the class balance and can dominate BCE.

In [ ]:
d5 = ladder[-1]["graph"]
train_graph, positive_edges, balanced_negatives = sample_link_split(d5, seed=SEED)
nodes = list(train_graph.nodes())
adjacency = nx.to_numpy_array(train_graph, nodelist=nodes, dtype=float)
embeddings = spectral_embeddings(adjacency, dimensions=8)
index = {node: i for i, node in enumerate(nodes)}
all_negatives = [edge for edge in nx.non_edges(train_graph) if edge not in positive_edges]
positive_scores = np.array([float(embeddings[index[i]] @ embeddings[index[j]]) for i, j in positive_edges])
balanced_scores = np.array([float(embeddings[index[i]] @ embeddings[index[j]]) for i, j in balanced_negatives])
all_scores = np.array([float(embeddings[index[i]] @ embeddings[index[j]]) for i, j in all_negatives])
balanced_loss = float(np.mean(np.concatenate([-np.log(sigmoid(positive_scores) + 1e-12), -np.log(1.0 - sigmoid(balanced_scores) + 1e-12)])))
swamped_loss = float(np.mean(np.concatenate([-np.log(sigmoid(positive_scores) + 1e-12), -np.log(1.0 - sigmoid(all_scores) + 1e-12)])))
print("balanced negatives", len(balanced_negatives), "loss", round(balanced_loss, 3))
print("all non-edges", len(all_negatives), "loss", round(swamped_loss, 3))

## Evaluate it + Practice

- Metric: pairwise edge AUC; compare it with a no-skill baseline such as majority class or random pair ranking.
- Sanity check: D1 should match the lesson arithmetic exactly before you trust D2-D5.
- Ablation: decode without balanced negative sampling
- Failure signals: BCE is dominated by absent pairs or high dot products are read as calibrated probabilities
- Baseline: score all candidate pairs randomly

Practice prompts:
1. Change one graph-ladder noise value and predict which rung should move most.

2. Add a second diagnostic printout that exposes one intermediate tensor for D3.

3. Replace the simple readout with another CPU-only NumPy readout and compare the metric curve.